# Mental rotation of 3-D objects: analysis

The measures of Shepard & Metzler (1971), from one or more runs' `*_trials.csv`:

- **Fig. 2**: mean reaction time of correct "same" responses as a function of angular difference, separately for picture-plane pairs (A) and depth pairs (B), with the fitted line; its slope gives the rate of "mental rotation" (the paper: roughly 60 deg/s).
- **Polynomial trend** per subject and rotation type, as the paper reports it (p. 703): the linear component tested against the deviations from linearity, and the quadratic and higher-order components tested against the within-angle error.
- **Errors** (the paper: 3.2% overall, 0.6-5.7% per subject) and the mean RT of correct "different" responses (the paper: 3.8 s; no angle is defined for these pairs).
- **Pure vs mixed blocks** (the paper: RTs no more than 20% lower in pure blocks), when the data have both.

Only correct responses enter the RT analyses, as in the paper. Set `ANALYSIS_RUN` (or `RUN` below) to a run folder, or to a folder holding several runs (all sessions of all subjects); by default the newest run under `data/` or `data-rehearsal/` is used.

Simulated data only exercises this notebook: a simulated subject's numbers are what was planted in it, not a replication.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
RUN = os.environ.get('ANALYSIS_RUN')
if RUN is None:
    runs = sorted(HERE.glob('data*/**/*_trials.csv'), key=lambda p: p.stat().st_mtime)
    RUN = runs[-1].parent if runs else None
print('run:', RUN)

In [ ]:
files = sorted(Path(RUN).rglob('*_trials.csv')) if RUN else []
trials = pd.concat([pd.read_csv(f).assign(source=f.name) for f in files], ignore_index=True) if files else pd.DataFrame()
if len(trials):
    trials['subject'] = trials['source'].str.extract(r'sub-([^_]+)_')[0]
    done = trials[trials['completed'].astype(bool)].copy()
    done['correct'] = done['outcome'] == 'CORRECT'
    done['rt_s'] = done['rt_ms'] / 1000.0
    print(f"{len(trials)} attempts, {len(done)} completed trials, subjects: {sorted(done['subject'].unique())}")
    print(done.groupby(['rotation', 'kind']).size().unstack())

## Errors

In [ ]:
if len(trials):
    err = 1 - done['correct'].mean()
    print(f'error rate overall: {100 * err:.1f}%  (paper: 3.2%)')
    print((100 * (1 - done.groupby('subject')['correct'].mean())).round(1).rename('error %'))
    print((100 * (1 - done.groupby(['rotation', 'kind'])['correct'].mean())).round(1).rename('error %'))

## Fig. 2: RT of correct "same" responses vs angular difference

In [ ]:
ROTATIONS = [('picture-plane', 'A  picture-plane pairs'), ('depth', 'B  depth pairs')]
fits = {}
if len(trials):
    same = done[(done['kind'] == 'same') & done['correct']]
    fig, axes = plt.subplots(2, 1, figsize=(5, 7), sharex=True)
    for ax, (rot, title) in zip(axes, ROTATIONS):
        d = same[same['rotation'] == rot]
        if len(d) == 0:
            ax.set_title(title + ' (no data)')
            continue
        m = d.groupby('angle_deg')['rt_s'].agg(['mean', 'sem', 'size'])
        ax.errorbar(m.index, m['mean'], yerr=m['sem'].fillna(0), fmt='o', color='black', ms=4)
        if d['angle_deg'].nunique() >= 2:
            slope, intercept, r, p, se = stats.linregress(d['angle_deg'], d['rt_s'])
            fits[rot] = dict(slope_s_per_deg=slope, intercept_s=intercept, r=r, p=p,
                             rate_deg_per_s=(1 / slope) if slope > 0 else np.nan, n=len(d))
            xs = np.array([0, 180])
            ax.plot(xs, intercept + slope * xs, color='#7f1d1d')
        ax.set_title(title)
        ax.set_ylabel('mean RT (s)')
        ax.set_xticks(range(0, 181, 20))
    axes[-1].set_xlabel('angle of rotation (degrees)')
    fig.tight_layout()
    print(pd.DataFrame(fits).T.round(4))

## Polynomial trend, per subject and rotation type (p. 703)

Angle is a 10-level factor, with orthogonal polynomial contrasts over its levels. The linear component is tested against the deviations from linearity, F(1, k-2); the quadratic, and the higher-order remainder, against the within-angle error.

In [ ]:
def trend(d):
    levels = np.sort(d['angle_deg'].unique())
    k, n = len(levels), len(d)
    if k < 4 or n <= k:
        return None
    grand = d['rt_s'].mean()
    g = d.groupby('angle_deg')['rt_s']
    means, counts = g.mean().reindex(levels).to_numpy(), g.size().reindex(levels).to_numpy()
    ss_between = float((counts * (means - grand) ** 2).sum())
    ss_within = float(((d['rt_s'] - g.transform('mean')) ** 2).sum())
    # orthogonal (count-weighted) polynomial contrasts, Gram-Schmidt over the levels
    x = (levels - levels.mean()) / levels.std()
    basis = []
    for deg in (0, 1, 2):
        v = x ** deg
        for b in basis:
            v = v - (counts * v * b).sum() / (counts * b * b).sum() * b
        basis.append(v)
    ss = [float((counts * means * b).sum() ** 2 / (counts * b * b).sum()) for b in basis[1:]]
    ms_dev = (ss_between - ss[0]) / (k - 2)
    ms_within = ss_within / (n - k)
    higher = max(ss_between - ss[0] - ss[1], 0.0) / (k - 3)
    return pd.Series({
        'n': n, 'F_linear_vs_deviation': ss[0] / ms_dev,
        'p_linear': stats.f.sf(ss[0] / ms_dev, 1, k - 2),
        'p_quadratic': stats.f.sf(ss[1] / ms_within, 1, n - k),
        'p_higher_order': stats.f.sf(higher / ms_within, k - 3, n - k)})

if len(trials):
    rows = {(s, rot): trend(d) for (s, rot), d in same.groupby(['subject', 'rotation'])}
    table = pd.DataFrame({k: v for k, v in rows.items() if v is not None}).T
    print(table.round(4) if len(table) else 'not enough angles per subject for the trend test')

## "Different" pairs, and pure vs mixed blocks

In [ ]:
if len(trials):
    diff = done[(done['kind'] == 'different') & done['correct']]
    if len(diff):
        print(f"mean RT, correct 'different': {diff['rt_s'].mean():.2f} s  (paper: 3.8 s)")
        print(f"mean RT, correct 'same':      {same['rt_s'].mean():.2f} s")
    if done['block_type'].nunique() > 1:
        bt = same.assign(block=np.where(same['block_type'] == 'mixed', 'mixed', 'pure'))
        print(bt.groupby(['rotation', 'block'])['rt_s'].mean().unstack().round(3))
    else:
        print('one block type in these data:', done['block_type'].unique())